# 02b · Gold — métricas de negocio y features para el modelo

**Preguntas que se contestan en esta capa:**
1. ¿Qué categorías y tallas se devuelven más? (para ajustar las guías de talla y las fotos)
2. ¿Cómo evoluciona la tasa de devolución semana a semana?
3. ¿Qué clientes devuelven más?

Y se prepara la tabla de entrada del modelo: **una fila por línea de pedido, con sus features y la etiqueta**.

## Qué pasa en este notebook

Gold es la capa que se consume: tablas ya agregadas, listas para un tablero o para entrenar un modelo. Se construye **después del `MERGE`** porque necesita las devoluciones aplicadas.

Hay dos tipos de tablas:

- **Tablas de métricas** (`resumen_devoluciones_categoria_talla`, `resumen_devoluciones_semanal`, `clientes_riesgo_devolucion`). Contestan preguntas de negocio *sobre lo que ya pasó*. Dicen qué categorías y qué clientes devuelven más **en promedio**, pero no dicen cuál línea concreta se va a devolver.
- **Tabla de features** (`features_linea_devolucion`). Es la base del modelo de Machine Learning. Aquí aparece la razón de ser del ML: la decisión se quiere tomar **por línea, antes de que ocurra la devolución**, combinando muchas variables a la vez (talla, categoría, descuento, historial del cliente), algo que una tabla agregada no puede hacer.



**En esta celda** se hereda la configuración compartida.

In [0]:
%run ./00_Setup/00_Configuracion

**En esta celda** se importan las funciones y tipos de PySpark y, además, **matplotlib** (`plt`), la librería de gráficas con la que más abajo se visualizan las tres respuestas de negocio de Gold.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast
import matplotlib.pyplot as plt

**En esta celda** se leen las tres tablas Silver y se construye una vista enriquecida: cada línea con la categoría, talla y color del producto y el segmento y ciudad del cliente. Los `join` usan `broadcast()` en las dimensiones, que son pequeñas, para evitar mover la tabla grande por la red.

In [0]:
df_lineas = spark.table(tbl(SCHEMA_SILVER, "lineas_pedido"))
df_productos = spark.table(tbl(SCHEMA_SILVER, "productos")).select("producto_id", "categoria", "talla", "color", "precio_lista")
df_clientes = spark.table(tbl(SCHEMA_SILVER, "clientes")).select("cliente_id", "segmento", "ciudad")

df_enriquecido = (
    df_lineas
    .join(broadcast(df_productos), on="producto_id", how="inner")
    .join(broadcast(df_clientes), on="cliente_id", how="inner")
)
print(f"Lineas enriquecidas: {df_enriquecido.count():,}")

**En esta celda** se contesta la primera pregunta: la tasa de devolución por categoría y talla. Se agrupa y se calculan `lineas`, `devueltas` y la tasa (`devueltas * 100 / lineas`). Es la tabla que ayuda a decidir en qué tallas y categorías conviene mejorar la guía de tallas o las fotos.

In [0]:
df_cat_talla = (
    df_enriquecido
    .groupBy("categoria", "talla")
    .agg(F.count("*").alias("lineas"), F.sum("fue_devuelto").alias("devueltas"))
    .withColumn("tasa_devolucion_pct", F.round(F.col("devueltas") * 100 / F.col("lineas"), 1))
)
escribir_delta(df_cat_talla, SCHEMA_GOLD, "resumen_devoluciones_categoria_talla",
               "Tasa de devolucion por categoria y talla")
display(df_cat_talla.orderBy(F.desc("tasa_devolucion_pct")).limit(10))

**En esta celda** se grafica la primera respuesta de Gold: un mapa de calor con la tasa de devolución por categoría y talla, leído de la tabla `resumen_devoluciones_categoria_talla` con matplotlib. Los tonos más intensos señalan dónde conviene intervenir primero: ajustar las guías de talla y mejorar las fotos del producto.


In [0]:
df_grafico = spark.table(tbl(SCHEMA_GOLD, "resumen_devoluciones_categoria_talla")).toPandas()
matriz = df_grafico.pivot(index="categoria", columns="talla", values="tasa_devolucion_pct")
matriz = matriz[["XS", "S", "M", "L", "XL"]]

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(matriz.values, cmap="Reds", aspect="auto")
ax.set_xticks(range(matriz.shape[1]), matriz.columns)
ax.set_yticks(range(matriz.shape[0]), matriz.index)
umbral = matriz.values.max() * 0.65
for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        ax.text(j, i, f"{matriz.values[i, j]:.0f}%", ha="center", va="center",
                color="white" if matriz.values[i, j] > umbral else "black")
ax.set_title("Tasa de devolucion (%) por categoria y talla")
fig.colorbar(im, ax=ax, shrink=0.8, label="% devuelto")
plt.tight_layout()
plt.show()

**En esta celda** se contesta la segunda pregunta: la evolución semanal. `F.date_trunc("week", fecha)` lleva cada fecha al inicio de su semana. Se suman `lineas` y `devueltas` (son aditivas) y la tasa se calcula **después** de sumar.

In [0]:
df_semanal = (
    df_enriquecido
    .withColumn("semana", F.date_trunc("week", F.col("fecha")).cast(DateType()))
    .groupBy("semana")
    .agg(F.count("*").alias("lineas"), F.sum("fue_devuelto").alias("devueltas"))
    .withColumn("tasa_devolucion_pct", F.round(F.col("devueltas") * 100 / F.col("lineas"), 1))
)
escribir_delta(df_semanal, SCHEMA_GOLD, "resumen_devoluciones_semanal", "Evolucion semanal de la tasa de devolucion")
display(df_semanal.orderBy("semana"))

**En esta celda** se grafica la segunda respuesta de Gold: la evolución semanal de la tasa de devolución, leída de la tabla `resumen_devoluciones_semanal`. La línea punteada marca el promedio del periodo, para que las semanas anómalas salten a la vista.


In [0]:
df_grafico = spark.table(tbl(SCHEMA_GOLD, "resumen_devoluciones_semanal")).orderBy("semana").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_grafico["semana"], df_grafico["tasa_devolucion_pct"], marker="o", markersize=3, linewidth=1, color="tab:blue")
promedio = df_grafico["tasa_devolucion_pct"].mean()
ax.axhline(promedio, linestyle="--", color="gray", label=f"promedio: {promedio:.1f}%")
ax.set_title("Evolucion semanal de la tasa de devolucion")
ax.set_xlabel("Semana")
ax.set_ylabel("% devuelto")
ax.legend()
plt.tight_layout()
plt.show()

**En esta celda** se contesta la tercera pregunta: qué clientes devuelven más. Se agrupa por cliente y se agrega su segmento. Esta tabla sirve para análisis de negocio sobre lo ocurrido; **no** se usa como feature del modelo, porque su tasa se calcula con todo el historial (incluye la línea que se predice) y filtraría la etiqueta.

In [0]:
df_clientes_riesgo = (
    df_enriquecido
    .groupBy("cliente_id", "segmento")
    .agg(F.count("*").alias("lineas"), F.sum("fue_devuelto").alias("devueltas"))
    .withColumn("tasa_devolucion_pct", F.round(F.col("devueltas") * 100 / F.col("lineas"), 1))
)
escribir_delta(df_clientes_riesgo, SCHEMA_GOLD, "clientes_riesgo_devolucion",
               "Resumen de devoluciones por cliente. Solo para analisis, no es feature del modelo")
display(df_clientes_riesgo.filter("lineas >= 5").orderBy(F.desc("tasa_devolucion_pct")).limit(10))

**En esta celda** se grafica la tercera respuesta de Gold: los 10 clientes con mayor tasa de devolución (mínimo 5 líneas, para no premiar a quien compró una sola vez), leídos de la tabla `clientes_riesgo_devolucion` y coloreados por segmento. Es la vista de negocio sobre lo ocurrido: como dice la descripción de la tabla, sirve para análisis, no como feature del modelo.

In [0]:
df_grafico = (
    spark.table(tbl(SCHEMA_GOLD, "clientes_riesgo_devolucion"))
    .filter("lineas >= 5")
    .orderBy(F.desc("tasa_devolucion_pct"))
    .limit(10)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 4.5))
colores = {"Nuevo": "tab:red", "Recurrente": "tab:orange", "VIP": "tab:blue"}
barras = ax.barh(
    [f"cliente {int(c)} ({s})" for c, s in zip(df_grafico["cliente_id"], df_grafico["segmento"])],
    df_grafico["tasa_devolucion_pct"],
    color=[colores.get(s, "gray") for s in df_grafico["segmento"]],
)
ax.bar_label(barras, fmt="%.1f%%", padding=3)
ax.invert_yaxis()
ax.set_xlabel("% devuelto")
ax.set_title("Top 10 clientes con mayor tasa de devolucion")
plt.tight_layout()
plt.show()

**En esta celda** se construye la tabla de features del modelo, con el historial de cada cliente calculado **solo hacia atrás**. Se usa una función de ventana (`Window`) particionada por cliente y ordenada por el día de la compra, con `rangeBetween(Window.unboundedPreceding, -35)`: para cada línea, solo entran las líneas del mismo cliente que ocurrieron **al menos 35 días antes**. De ahí salen tres features:

- `num_lineas_previas`: cuántas compras anteriores (ya conocidas) tiene el cliente.
- `num_devueltas_previas`: cuántas de ellas se devolvieron.
- `tasa_devolucion_previa`: la razón entre ambas. Se divide por `max(num_lineas_previas, 1)` para evitar la división por cero; un cliente sin historial queda con tasa 0 y `num_lineas_previas = 0`, y el modelo puede distinguir ese caso.

**Por qué 35 días:** una devolución puede tardar hasta 35 días en llegar. Pasado ese plazo, la etiqueta de una línea antigua ya es definitiva y se puede usar como historial sin "mirar el futuro".

In [0]:
w_cliente_pasado = (
    Window.partitionBy("cliente_id")
    .orderBy("_dia")
    .rangeBetween(Window.unboundedPreceding, -MAX_DIAS_HASTA_DEVOLUCION)
)

df_features = (
    df_enriquecido
    .withColumn("_dia", F.datediff(F.col("fecha"), F.lit(FECHA_INICIO).cast(DateType())))
    .withColumn("num_lineas_previas", F.count("linea_id").over(w_cliente_pasado))
    .withColumn("num_devueltas_previas", F.coalesce(F.sum("fue_devuelto").over(w_cliente_pasado), F.lit(0)))
    .withColumn("tasa_devolucion_previa",
                F.col("num_devueltas_previas") / F.greatest(F.col("num_lineas_previas"), F.lit(1)))
    .select("linea_id", "cliente_id", "fecha", "categoria", "talla", "color", "segmento",
            "precio_lista", "descuento_pct", "cantidad",
            "num_lineas_previas", "num_devueltas_previas", "tasa_devolucion_previa", "fue_devuelto")
)

escribir_delta(df_features, SCHEMA_GOLD, "features_linea_devolucion",
               "Features por linea de pedido con historial del cliente sin fuga de datos, y etiqueta fue_devuelto")
display(df_features.orderBy(F.desc("num_lineas_previas")).limit(10))

**En esta celda** se hace una verificación final de Gold: cuántas filas tiene cada tabla y la tasa de devolución global, que es el número base con el que el modelo tiene que competir.

In [0]:
print("=" * 60)
print("VERIFICACION GOLD")
print("=" * 60)
for tabla in ["resumen_devoluciones_categoria_talla", "resumen_devoluciones_semanal",
              "clientes_riesgo_devolucion", "features_linea_devolucion"]:
    print(f"  {tabla:<42} {spark.table(tbl(SCHEMA_GOLD, tabla)).count():>8,} registros")

tasa_global = spark.table(tbl(SCHEMA_GOLD, "features_linea_devolucion")).agg(F.avg("fue_devuelto")).first()[0]
print(f"\nTasa global de devolucion (fue_devuelto = 1): {tasa_global:.1%}")

Gold · Métricas y features